# RK1828 Whisper-base Speech Recognition Demo


OpenAI [whisper-base](https://github.com/openai/whisper), Chinese/English
recognition. Split into three models: **encoder** + **first-token decoder
(decode0)** + **autoregressive decoder (decode1)**. Measured RTF 0.073 on the
Chinese sample (~14x realtime).

| Step | Content |
|---|---|
| 1 | Download models and sample audio (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Pick language and audio |
| 4 | Transcribe |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("whisper-base-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/whisper_base/demo_x86_64.tar.gz"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "demos/whisper_base/model/long_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/mel_128_filters.txt", "demos/whisper_base/model/mel_128_filters.txt"),
    ("whisper-base-rknn3-rk1828", "model/mel_80_filters.txt", "demos/whisper_base/model/mel_80_filters.txt"),
    ("whisper-base-rknn3-rk1828", "model/position_embed.bin", "demos/whisper_base/model/position_embed.bin"),
    ("whisper-base-rknn3-rk1828", "model/prompt_embed.bin", "demos/whisper_base/model/prompt_embed.bin"),
    ("whisper-base-rknn3-rk1828", "model/prompt_embed_en.bin", "demos/whisper_base/model/prompt_embed_en.bin"),
    ("whisper-base-rknn3-rk1828", "model/prompt_embed_zh.bin", "demos/whisper_base/model/prompt_embed_zh.bin"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "demos/whisper_base/model/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "demos/whisper_base/model/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/token_embed.bin", "demos/whisper_base/model/token_embed.bin"),
    ("whisper-base-rknn3-rk1828", "model/vocab_en.txt", "demos/whisper_base/model/vocab_en.txt"),
    ("whisper-base-rknn3-rk1828", "model/vocab_zh.txt", "demos/whisper_base/model/vocab_zh.txt"),
    ("whisper-base-rknn3-rk1828", "model/whisper_decode0_base.rknn", "demos/whisper_base/model/whisper_decode0_base.rknn"),
    ("whisper-base-rknn3-rk1828", "model/whisper_decode0_base.weight", "demos/whisper_base/model/whisper_decode0_base.weight"),
    ("whisper-base-rknn3-rk1828", "model/whisper_decode1_base.rknn", "demos/whisper_base/model/whisper_decode1_base.rknn"),
    ("whisper-base-rknn3-rk1828", "model/whisper_decode1_base.weight", "demos/whisper_base/model/whisper_decode1_base.weight"),
    ("whisper-base-rknn3-rk1828", "model/whisper_encoder_base.rknn", "demos/whisper_base/model/whisper_encoder_base.rknn"),
    ("whisper-base-rknn3-rk1828", "model/whisper_encoder_base.weight", "demos/whisper_base/model/whisper_encoder_base.weight"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "datasets/asr/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "datasets/asr/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "datasets/asr/long_en.wav"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "datasets/asr/sensevoice_zh.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "datasets/asr/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "datasets/asr/asr_example_hotword.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "datasets/asr/aligner_demo.wav"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/whisper_base/demo_x86_64.tar.gz", "demos/whisper_base")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Pick language and audio

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_lang = widgets.Dropdown(options=[("Chinese", "zh"), ("English", "en")],
                          value="zh", description="Language:")
w_wav = widgets.Dropdown(options=sorted(glob.glob("datasets/asr/*.wav")),
                         description="Audio:", layout=widgets.Layout(width="60%"))
if any(o.endswith('/test_zh.wav') for o in w_wav.options):
    w_wav.value = [o for o in w_wav.options if o.endswith('/test_zh.wav')][0]
display(w_lang, w_wav)
print("Then keep running downwards ▼")

Listen to the audio first:

In [ ]:
from nb_demo import show_audio

show_audio(w_wav.value)

## 4. Transcribe

In [ ]:
import os

DEMO = "demos/whisper_base"
wav = os.path.abspath(w_wav.value)
run_demo(["./rknn_whisper_raw_demo",
          "model/whisper_encoder_base.rknn", "model/whisper_encoder_base.weight",
          "model/whisper_decode0_base.rknn", "model/whisper_decode0_base.weight",
          "model/whisper_decode1_base.rknn", "model/whisper_decode1_base.weight",
          w_lang.value, wav, "model/token_embed.bin",
          "0xff", "0xff", "0xff", "--model-type", "base"], DEMO)

## Summary

- Whisper's autoregressive decoder is split into a "first token" model and a
  "later tokens" model — a common NPU optimization for Transformer decoders
  (the first token needs no KV cache).
- Long audio (e.g. `long_en.wav`) is chunked automatically.